# Rozpoznawania arytmii na podstawie sygnału EKG

Rozpatrujemy trzy możliwe podejścia do rozpoznawania wybranych zaburzeń rytmu serca:

1. **podejście klasyczne** - analiza odstępów RR i proste reguły,
2. **klasyczne uczenie maszynowe** - klasyfikacja na podstawie cech,
3. **deep learning** - klasyfikacja na podstawie surowych lub przefiltrowanych okien sygnału.

## Wspólny schemat przetwarzania

Niezależnie od wybranej metody, początkowe etapy pipeline'u są podobne:

```text
dane EKG
→ wybór kanału
→ wizualizacja sygnału
→ filtracja
→ piki R / adnotacje
→ segmentacja lub analiza RR
→ etykiety
→ model / reguły
→ ewaluacja
```

Różnica między metodami polega głównie na tym, jakie informacje są wykorzystywane:

| Podejście | Dane wejściowe | Interpretowalność | Trudność |
|---|---|---:|---:|
| Analiza RR | odstępy między pikami R | wysoka | niska |
| ML na cechach | cechy rytmu i morfologii | średnia/wysoka | średnia |
| Deep learning | okna sygnału EKG | niższa | wysoka |

## Podejście I - klasyczna analiza odstępów RR

### Założenie

Wiele zaburzeń rytmu objawia się zmianą regularności pracy serca. Dlatego pierwszym, prostym podejściem może być analiza odstępów RR, czyli czasów między kolejnymi pikami R.

### Dane wejściowe

- pozycje pików R,
- częstotliwość próbkowania,
- opcjonalnie etykiety uderzeń z bazy.

### Pipeline

```text
EKG
→ piki R
→ odstępy RR
→ chwilowe HR
→ lokalna średnia RR
→ wykrywanie odstępów nietypowych
→ porównanie z adnotacjami
```

### Przykładowe cechy

- `RR_i` - aktualny odstęp RR,
- `RR_prev` - poprzedni odstęp RR,
- `RR_next` - następny odstęp RR,
- `HR = 60 / RR`,
- odchylenie od lokalnej średniej,
- stosunek aktualnego RR do mediany lokalnej.

### Zalety

- proste,
- interpretowalne,
- dobrze pasuje do analizy rytmu,
- dobre jako baseline.

### Ograniczenia

- nie analizuje kształtu zespołu QRS,
- może nie rozróżniać typów arytmii,
- nie wykryje dobrze zaburzeń widocznych głównie w morfologii uderzenia.

## 4. Podejście II - klasyczne uczenie maszynowe na cechach

### Założenie

Sama analiza RR może być niewystarczająca, ponieważ nie uwzględnia morfologii uderzenia. Dlatego drugim krokiem może być klasyfikacja uderzeń na podstawie cech liczonych z okien sygnału EKG.

### Dane wejściowe

Dla każdego uderzenia:

- okno sygnału wokół piku R,
- etykieta uderzenia,
- cechy rytmu,
- cechy morfologiczne,
- opcjonalnie cechy częstotliwościowe.

### Pipeline

```text
EKG
→ filtracja
→ piki R / adnotacje
→ okna wokół R
→ ekstrakcja cech
→ skalowanie
→ klasyfikator ML
→ ewaluacja
```

### Przykładowe modele

- Logistic Regression,
- k-NN,
- SVM,
- Random Forest,
- Gradient Boosting,
- XGBoost/LightGBM, jeśli projekt ma być bardziej zaawansowany.

## Podejście III - deep learning na oknach sygnału

### Założenie

W deep learningu nie trzeba ręcznie projektować cech. Model może dostać bezpośrednio fragment sygnału EKG wokół piku R i samodzielnie nauczyć się reprezentacji.

Najbardziej naturalnym modelem dla takiego problemu jest **1D CNN**, czyli jednowymiarowa sieć konwolucyjna.

### Dane wejściowe

Dla każdego uderzenia:

```text
okno EKG wokół piku R
```

Przykład:

- 200 ms przed R,
- 400 ms po R,
- przy fs = 360 Hz daje to około 216 próbek.

Kształt danych dla CNN:

```text
liczba_uderzeń × liczba_próbek × liczba_kanałów
```

Dla jednego kanału:

```text
N × 216 × 1
```

### Pipeline

```text
EKG
→ filtracja
→ piki R / adnotacje
→ okna wokół R
→ normalizacja każdego okna
→ 1D CNN
→ ewaluacja
```

### Zalety

- model uczy się cech automatycznie,
- może uchwycić złożoną morfologię uderzeń,
- dobre rozszerzenie projektu ML.

### Ograniczenia

- wymaga więcej danych,
- trudniejszy do interpretacji,
- łatwo o overfitting,
- wymaga ostrożnego podziału danych.

## Metryki oceny

W klasyfikacji arytmii klasy są zwykle niezbalansowane. Uderzeń prawidłowych jest często dużo więcej niż nieprawidłowych. Dlatego **accuracy** może być mylące.

Warto raportować:

| Metryka | Znaczenie |
|---|---|
| **accuracy** | ogólna skuteczność, ale może być myląca przy niezbalansowanych klasach |
| **precision** | jaki odsetek przewidzianych arytmii był rzeczywiście arytmią |
| **recall / sensitivity** | jaki odsetek rzeczywistych arytmii został wykryty |
| **specificity** | jaki odsetek prawidłowych uderzeń poprawnie rozpoznano jako prawidłowe |
| **F1-score** | kompromis między precision i recall |
| **confusion matrix** | liczba TP, FP, TN, FN |
| **ROC-AUC / PR-AUC** | ocena modelu przy różnych progach decyzyjnych |

Dla projektu wykrywania nieprawidłowych uderzeń szczególnie ważny jest **recall dla klasy nieprawidłowej**, ponieważ pokazuje, ile potencjalnych zaburzeń model wykrył.